In [ ]:
import warnings
from pathlib import Path
import sys
from contextlib import ExitStack
from importlib import resources
 
try:
    from proteomicscopylot.notebook_functions import *
    print("Functions source: pip package/local package proteomicscopylot.notebook_functions")
except ModuleNotFoundError:
    project_root = Path.cwd().resolve()
    if not (project_root / "proteomicscopylot" / "notebook_functions.py").exists():
        project_root = project_root.parent
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))
    from proteomicscopylot.notebook_functions import *
    print(f"Functions source: local checkout ({project_root / 'proteomicscopylot' / 'notebook_functions.py'})")
 
warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")


In [ ]:
_DATA_FILES = {
    "data": "20260511_144120_20240822_JM_HEK293_WP_Report_PG.tsv",
    "data3": "20260511_144129_20240822_JM_HEK293_PP_Report_collapsed.tsv",
    "data4": "phosprot_filtered_data_full.csv",
    "meta": "protein_metadata_full.csv",
    "meta2": "phospho_metadata_full.csv",
}

_DATA_RESOURCE_STACK = ExitStack()
_DATA_PATHS = {}
_DATA_SOURCE = None

for _package_name, _subdir in [
    ("proteomicscopylot.example_data", None),
    ("proteomicscopylot.data.example_data", None),
    ("proteomicscopylot", "example_data"),
]:
    try:
        _root = resources.files(_package_name)
        if _subdir is not None:
            _root = _root / _subdir
        _candidate_paths = {key: _root / filename for key, filename in _DATA_FILES.items()}
        if all(path.is_file() for path in _candidate_paths.values()):
            _DATA_PATHS = {
                key: _DATA_RESOURCE_STACK.enter_context(resources.as_file(path))
                for key, path in _candidate_paths.items()
            }
            _DATA_SOURCE = f"pip package {_package_name}" + (f"/{_subdir}" if _subdir else "")
            break
    except ModuleNotFoundError:
        pass

if _DATA_SOURCE is None:
    project_root = Path.cwd().resolve()
    if not (project_root / "proteomicscopylot" / "example_data").exists():
        project_root = project_root.parent
    _root = project_root / "proteomicscopylot" / "example_data"
    _DATA_PATHS = {key: _root / filename for key, filename in _DATA_FILES.items()}
    _missing = [str(path) for path in _DATA_PATHS.values() if not path.exists()]
    if _missing:
        raise FileNotFoundError("Missing example data files: " + ", ".join(_missing))
    _DATA_SOURCE = f"local checkout ({_root})"

print(f"Data source: {_DATA_SOURCE}")

data = read_data(_DATA_PATHS["data"])
data3 = read_data(_DATA_PATHS["data3"])
data4 = read_data(_DATA_PATHS["data4"])
meta = read_data(_DATA_PATHS["meta"])
meta2 = read_data(_DATA_PATHS["meta2"])


In [ ]:
org_data = inverse_log2_transform_data(data, meta)
org_data3 = inverse_log2_transform_data(data3, meta2)
org_data4 = inverse_log2_transform_data(data4, meta2)

conditions = ["WT", "GRK"]

data_by_protein = data.set_index("ProteinNames", drop=False)
data3_by_site = data3.set_index("PTM_Collapse_key", drop=False)
data4_by_phosprot = data4.set_index("Phosphoprotein", drop=False)

protein_id = data["ProteinNames"].dropna().astype(str).iloc[0]
phosphosite_id = data3["PTM_Collapse_key"].dropna().astype(str).iloc[0]
phospho_protein_id = data3["Protein_group"].dropna().astype(str).iloc[0]
phosphoprotein_id = data4["Phosphoprotein"].dropna().astype(str).iloc[0]

print(f"Protein example: {protein_id}")
print(f"Phosphosite example: {phosphosite_id}")
print(f"Phospho protein example: {phospho_protein_id}")


# Protein


## Completeness


In [ ]:
fig = missing_value_plot(data, meta)

In [ ]:
fig = missing_value_heatmap(data, meta, id=False)

In [ ]:
missing_protein_table(data, meta, id=False)

In [ ]:
missing_sample_table(data, meta, id=False)

## QC Pipeline


In [ ]:
fig = coverage_plot(data, meta, id=False)

In [ ]:
fig = coverage_plot_summary(data, meta)

In [ ]:
fig = histo_int(data, meta)

In [ ]:
fig = boxplot_int(data, meta)

In [ ]:
fig = cov_plot(org_data, meta)

In [ ]:
fig = pca_plot(data, meta)

In [ ]:
fig = abundance_plot(org_data, meta, workflow="Protein")

In [ ]:
fig = corr_plot(org_data, meta, id=False)

## Volcano Plot


In [ ]:
volcano_data = volcano_data_f(data, meta, "WT", "GRK", workflow="Protein")
volcano_data.head()

In [ ]:
fig = volcano_plot(volcano_data, "WT", "GRK")
fig.show()

## Single Protein


In [ ]:
fig = boxplot_int_single_prot(data_by_protein, meta, protein_id, dots=True)

In [ ]:
fig = compare_prot_line(data, meta, conditions, [protein_id], workflow="Protein", id=False)

## Comparison


In [ ]:
fig = correlation_xy_intensity(data, meta, "WT", "GRK")

In [ ]:
fig = overlap_plot_condition(data_by_protein, meta, "WT", "GRK")

# Phospho


## Phospho-Specific


In [ ]:
fig = simple_phos_site_plot(data3)

In [ ]:
fig = phossite_coverage_plot(data3, meta2, id=False)

In [ ]:
fig = sty_plot(data3)

In [ ]:
fig = phossite_dist(data4, cutoff=0.01)

## Completeness


In [ ]:
fig = missing_value_plot(data3, meta2)

In [ ]:
fig = missing_value_heatmap(data3, meta2, id=False)

In [ ]:
missing_protein_table(data3, meta2, id=False)

In [ ]:
missing_sample_table(data3, meta2, id=False)

## QC Pipeline


In [ ]:
fig = coverage_plot(data3, meta2, id=False)

In [ ]:
fig = coverage_plot_summary(data3, meta2)

In [ ]:
fig = histo_int(data3, meta2)

In [ ]:
fig = boxplot_int(data3, meta2)

In [ ]:
fig = cov_plot(org_data3, meta2)

In [ ]:
fig = pca_plot(data3, meta2)

In [ ]:
fig = abundance_plot(org_data3, meta2, workflow="Phosphosite")

In [ ]:
fig = corr_plot(org_data3, meta2, id=False)

## Volcano Plot


In [ ]:
volcano_data = volcano_data_f(data3, meta2, "WT", "GRK", workflow="Phosphosite")
volcano_data.head()

In [ ]:
fig = volcano_plot(volcano_data, "WT", "GRK")
fig.show()

## Single Protein


In [ ]:
fig = boxplot_int_single_prot(data3_by_site, meta2, phosphosite_id, dots=True)

In [ ]:
fig = compare_prot_line(data3, meta2, conditions, [phosphosite_id], workflow="Phosphosite", id=False)

In [ ]:
fig = phossite_on_prot_regulation(data3, meta2, phospho_protein_id, conditions=conditions, id=False)

## Comparison


In [ ]:
fig = correlation_xy_intensity(data3, meta2, "WT", "GRK")

In [ ]:
fig = overlap_plot_condition(data3_by_site, meta2, "WT", "GRK")

# Phosphoprotein


## Completeness


In [ ]:
fig = missing_value_plot(data4, meta2)

In [ ]:
fig = missing_value_heatmap(data4, meta2, id=False)

In [ ]:
missing_protein_table(data4, meta2, id=False)

In [ ]:
missing_sample_table(data4, meta2, id=False)

## QC Pipeline


In [ ]:
fig = coverage_plot(data4, meta2, id=False)

In [ ]:
fig = coverage_plot_summary(data4, meta2)

In [ ]:
fig = histo_int(data4, meta2)

In [ ]:
fig = boxplot_int(data4, meta2)

In [ ]:
fig = cov_plot(org_data4, meta2)

In [ ]:
fig = pca_plot(data4, meta2)

In [ ]:
fig = abundance_plot(org_data4, meta2, workflow="Phosphoprotein")

In [ ]:
fig = corr_plot(org_data4, meta2, id=False)

## Volcano Plot


In [ ]:
volcano_data = volcano_data_f(data4, meta2, "WT", "GRK", workflow="Phosphoprotein")
volcano_data.head()

In [ ]:
fig = volcano_plot(volcano_data, "WT", "GRK")
fig.show()

## Single Protein


In [ ]:
fig = boxplot_int_single_prot(data4_by_phosprot, meta2, phosphoprotein_id, dots=True)

In [ ]:
fig = compare_prot_line(data4, meta2, conditions, [phosphoprotein_id], workflow="Phosphoprotein", id=False)

## Comparison


In [ ]:
fig = correlation_xy_intensity(data4, meta2, "WT", "GRK")

In [ ]:
fig = overlap_plot_condition(data4_by_phosprot, meta2, "WT", "GRK")